# 第1回 いまのAIは何ができて、何ができないか — 実習ノートブック

**このノートブックは第1回だけで完結します。** プログラミングの経験は要りません。

やること:
1. AIが文をどう区切っているかを見る(トークン)
2. 次の語の確率を表示する(AIの正体)
3. 温度を変えて、出力のばらつきを確かめる

所要時間:実習 約20分 / 課題 約40分

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。
最初のセルは1〜2分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。1〜2分かかります)
!pip install -q transformers japanize-matplotlib
import warnings, torch
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt, japanize_matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
plt.rcParams["font.size"] = 12
print("準備できました")

In [ ]:
# 今日使うモデルを読み込む(初回は1分ほどダウンロードします)
# GPT-2:2019年の小さなモデル。中身を観察するにはこのくらいが分かりやすい。
トークナイザ = AutoTokenizer.from_pretrained("gpt2")
モデル = AutoModelForCausalLM.from_pretrained("gpt2")
モデル.eval()
print("読み込み完了 / 語彙数:", トークナイザ.vocab_size)

## 1. AIが見ているのは「文字」ではない

AIに「strawberry に r はいくつ?」と聞くと間違えることがあります。
理由は、AIが文字を1つずつ見ていないからです。

文を **トークン** という塊に区切ってから、塊ごとに番号をつけて処理しています。

In [ ]:
# ===== ① AIは文字をどう区切っているか =====
def 区切る(文字列):
    トークン = トークナイザ.tokenize(文字列)
    # Ġ は「単語の前の空白」を表す記号。読みやすいように置きかえる
    見やすく = [t.replace("Ġ", "␣") for t in トークン]
    print(f"{文字列!r}")
    print(f"   {len(文字列)}文字 → {len(トークン)}トークン : {見やすく}\n")

for s in ["strawberry", "cat", "caterpillar", "artificial intelligence", "こんにちは", "人工知能"]:
    区切る(s)

print("strawberry は st / raw / berry の3つに見えています。")
print("だから「r は何個?」と聞かれても、数える対象がそもそも見えていません。")

In [ ]:
# 文字数とトークン数は比例しない — 図で見る
語 = ["cat", "caterpillar", "strawberry", "intelligence", "artificial intelligence",
      "こんにちは", "人工知能", "機械学習"]
文字数 = [len(w) for w in 語]
トークン数 = [len(トークナイザ.tokenize(w)) for w in 語]

plt.figure(figsize=(7, 4.5))
plt.scatter(文字数, トークン数, s=90, color="#3A46A8", zorder=3)
for w, x, y in zip(語, 文字数, トークン数):
    plt.annotate(w, (x, y), xytext=(5, 5), textcoords="offset points", fontsize=10)
plt.xlabel("文字数"); plt.ylabel("トークン数")
plt.title("文字数とトークン数は比例しない")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("右下(文字数が多いのにトークンが少ない)= よく使われる語")
print("左上(文字数が少ないのにトークンが多い)= 日本語や珍しい語")

## 2. AIの正体 — 次の1語を確率で選んでいるだけ

大規模言語モデルがやっているのは、**文の続きとして次に来る語の確率を計算し、そこから1つ選ぶこと**。それだけです。
これを何百回もくり返すと、文章になります。

下のセルで、実際の確率を見てみましょう。

In [ ]:
# ===== ② AIの正体:次の1語を確率で選ぶ =====
def 次の語(文, 上位=8, 図=True):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        出力 = モデル(ids).logits[0, -1]      # 最後の位置の予測だけ見る
    確率 = torch.softmax(出力, dim=-1)
    上 = torch.topk(確率, 上位)
    語 = [トークナイザ.decode([i]).strip() or "␣" for i in 上.indices]
    値 = [float(v) * 100 for v in 上.values]
    print(f"「{文}」の次に来る語\n")
    for w, v in zip(語, 値):
        print(f"   {w:<14}{v:5.1f}%  {'■' * int(v)}")
    print(f"\n   上位{上位}語の合計 {sum(値):.1f}%  ← 残りは他の約5万語に分かれています")
    if 図:
        plt.figure(figsize=(7, 3.4))
        plt.barh(語[::-1], 値[::-1], color="#3A46A8")
        plt.xlabel("確率(%)"); plt.title(f"「{文}」の次に来る語")
        plt.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()
    return 語, 値

次の語("The sky is")

> **「事実かどうか」ではなく「続きとしてありそうか」で選ばれています。**
> `The sky is` の次に `falling`(落ちてくる)が高い確率で出るのは、英語に「空が落ちてくる」という言い回しがあるからです。

In [ ]:
# 演習・問2:フランスの首都は?
次の語("The capital of France is")

print("\n【考えてみてください】")
print("Paris は何位で何%でしたか。予想より高かったですか、低かったですか。")
print("ヒント:GPT-2 は2019年の小さなモデルです。第10回で「大きくすると何が変わるか」を扱います。")

## 3. ばらつきの正体 — 温度

同じ質問をしても毎回答えが違うのは、故障ではありません。
確率が出たあと「どう選ぶか」を決める **温度** という設定があるからです。

In [ ]:
# ===== ③ ばらつきの正体:温度 =====
def 温度を変える(文, 温度リスト=(0.2, 0.7, 1.0, 1.5), 回数=30, seed=0):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        出力 = モデル(ids).logits[0, -1]
    print(f"「{文}」の次の語を {回数}回 引いてみる\n")
    print(f"{'温度':>6}{'1位の確率':>12}{'上位5語の合計':>15}{'出た語の種類':>14}")
    for T in 温度リスト:
        確率 = torch.softmax(出力 / T, dim=-1)
        上 = torch.topk(確率, 5)
        g = torch.Generator().manual_seed(seed)
        引いた = torch.multinomial(確率, 回数, replacement=True, generator=g)
        print(f"{T:>6}{float(上.values[0])*100:>11.1f}%{float(上.values.sum())*100:>14.1f}%"
              f"{len(set(引いた.tolist())):>12}種類")
    print("\n温度が低い → 1位に集中して、いつも同じ語が出る")
    print("温度が高い → 確率がならされて、いろいろな語が出る")

温度を変える("The sky is")

In [ ]:
# 実際に文章を作らせて比べる(温度を変えて3回ずつ)
def 文を作る(文, 温度, 回数=3, 長さ=20):
    ids = トークナイザ.encode(文, return_tensors="pt")
    print(f"--- 温度 {温度} ---")
    for k in range(回数):
        torch.manual_seed(k)
        出力 = モデル.generate(ids, max_new_tokens=長さ, do_sample=True,
                               temperature=温度, top_k=50,
                               pad_token_id=トークナイザ.eos_token_id,
                               attention_mask=torch.ones_like(ids))
        print("  " + トークナイザ.decode(出力[0], skip_special_tokens=True).replace("\n", " "))
    print()

for T in [0.2, 0.7, 1.5]:
    文を作る("The best way to learn is", T)

print("温度0.2 は似た文ばかり、1.5 は毎回違うが壊れやすい、を確かめてください。")

## 4. 自分で試す

下のセルを書きかえて、自分の言葉で確かめてください。**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題2用)=====
# ここを自由に書きかえて実行してください。

# (a) 好きな文字列を5つ入れて、文字数とトークン数を比べる
私の語 = ["あなたの名前", "好きな食べ物", "専門用語", "machine learning", "sushi"]
print(f"{'文字列':<26}{'文字数':>6}{'トークン数':>10}")
for w in 私の語:
    print(f"{w:<26}{len(w):>6}{len(トークナイザ.tokenize(w)):>10}")

# (b) 好きな英文の続きを見る
print()
次の語("My favorite food is", 上位=6, 図=False)

# (c) 温度を変えて比べる
print()
温度を変える("Artificial intelligence will")

## 5. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:生成AIを触って、今日の「線引き」で説明する
ChatGPT・Gemini・Claude など、使えるものでかまいません。次の3種類を試して結果を貼ってください。

- (a) **もっともらしさで足りる仕事** — 文章の要約、言い換え
- (b) **正確さが要る仕事** — 「ちょうど100文字で」、桁の多い計算、出典を挙げさせる
- (c) **同じ質問を3回くり返す**

それぞれ、今日の線引き(トークン/確率で選ぶ)で説明できるかを書いてください。
説明できない結果が出たら、それも書いてください。むしろ面白い発見です。

### 課題2:ノートブックを動かして報告する
- (a) 好きな文字列を5つトークン化し、文字数とトークン数の表を作る
- (b) 好きな英文の続きの確率を表示し、上位5語を貼る
- (c) 温度を3通り変えて出力を比べ、気づいたことを3行以上で書く

### 課題3:次の主張を批判する(300字以上)

> 「AIは strawberry の r の数すら間違える。この程度の知能で人間の仕事が奪われるという話は、明らかに誇張である。」

考える材料:
- 前半の**事実は正しい**。では、その理由は「知能が低いから」ですか?
- 「文字を数える」は、AIにとってどういう種類の課題ですか?
- AIが**得意**な仕事は何でしたか? 仕事への影響は、どちら側で起きますか?

**加点**:「では、どんな仕事が影響を受けやすいか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
うまくいかなかった結果も、なぜそうなったかを書いてあれば評価します。